# CATI — Singapore Smart City Traffic Analytics
## Context-Aware Traffic Intelligence Demo

End-to-end pipeline on live Singapore LTA camera feeds:

| Stage | Module | What it does |
|---|---|---|
| Detection | `CATIPipeline` | FiLM-conditioned YOLOv11 — weather/time/location aware |
| Tracking | `SingaporeTracker` | Direction-constrained ByteTrack per expressway |
| Re-ID | `VehicleReID` | OSNet-x0.25 cross-camera appearance matching |
| Analytics | `TrafficAnalytics` | Occupancy, LOS A–F, congestion score |
| Speed | `SpeedEstimator` | Inter-camera speed via GPS edge distances |
| Network | `CameraNetwork` | 90-camera expressway graph (75 edges) |

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_DIR = '/content/sg-smart-city-analytics'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Suhxs-Reddy/sg-smart-city-analytics.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
    for k in list(sys.modules.keys()):
        if k.startswith('src.'): del sys.modules[k]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

!pip install -q ultralytics torch torchvision scipy pillow opencv-python-headless torchreid

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}  ({gpu.total_memory/2**30:.1f} GB)')

MODEL_DIR   = '/content/drive/MyDrive/sg_smart_city/models'
FEATURE_DIR = '/content/drive/MyDrive/sg_smart_city/data/features'
YOLO_DIR    = '/content/drive/MyDrive/sg_smart_city/data/yolo_dataset'
OUTPUT_DIR  = '/content/drive/MyDrive/sg_smart_city/demo_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

from pathlib import Path
PHASE2_DIR  = Path(MODEL_DIR) / 'phase2'
YOLO_WEIGHTS = str(max(PHASE2_DIR.glob('**/best.pt'), key=lambda p: p.stat().st_mtime))
CATI_WEIGHTS = str(max(
    list(PHASE2_DIR.glob('**/cati_phase2_final.pt')) +
    list(PHASE2_DIR.glob('**/cati_phase2_epoch*.pt')),
    key=lambda p: p.stat().st_mtime
))
print(f'YOLO: {YOLO_WEIGHTS}')
print(f'CATI: {CATI_WEIGHTS}')

In [ ]:
# ── Cell 2: Camera Network Overview ───────────────────────────────────────
from src.analytics.camera_network import CameraNetwork
from src.analytics.camera_map import CameraMap

net = CameraNetwork()
print(net.summary())
print()
print('CTE corridor (S→N):')
for e in net.adjacent_pairs('CTE')[:6]:
    print(f'  {e.cam_a.camera_id} ({e.cam_a.area}) → {e.cam_b.camera_id} ({e.cam_b.area})  {e.distance_km:.2f} km')
print('  ...')

In [ ]:
# ── Cell 3: Load Pipeline ──────────────────────────────────────────────────
from src.inference import CATIPipeline

pipeline = CATIPipeline(
    yolo_weights=YOLO_WEIGHTS,
    cati_weights=CATI_WEIGHTS,
    feature_dir=FEATURE_DIR,
    device='cuda',
    conf=0.25,
    use_neck_film=True,
)
print('Pipeline loaded.')

In [ ]:
# ── Cell 4: Fetch Live LTA Frames ─────────────────────────────────────────
import urllib.request, json, cv2, numpy as np
from datetime import datetime, timezone, timedelta
from pathlib import Path

SGT = timezone(timedelta(hours=8))
SELECTED = ['1001','1701','4701','4710','5794','6701','7791','8701','9701']
frames = {}
timestamp = datetime.now(SGT).isoformat()

# ── Try live LTA API ──
# LTA image servers require Referer: https://data.gov.sg
try:
    req = urllib.request.Request(
        'https://api.data.gov.sg/v1/transport/traffic-images',
        headers={'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}
    )
    data = json.loads(urllib.request.urlopen(req, timeout=10).read())
    cameras = data['items'][0]['cameras']
    print(f'LTA API: {len(cameras)} cameras at {timestamp}')
    for cam in cameras:
        cid = cam['camera_id']
        if cid not in SELECTED:
            continue
        try:
            img_req = urllib.request.Request(
                cam['image'],
                headers={
                    'User-Agent': 'Mozilla/5.0',
                    'Referer': 'https://data.gov.sg/',
                    'Accept': 'image/jpeg,image/*',
                }
            )
            resp = urllib.request.urlopen(img_req, timeout=8)
            img_bytes = np.frombuffer(resp.read(), np.uint8)
            img = cv2.imdecode(img_bytes, cv2.IMREAD_COLOR)
            if img is not None:
                frames[cid] = {'image': img,
                               'lat': cam['location']['latitude'],
                               'lon': cam['location']['longitude']}
                print(f'  {cid}: {img.shape[1]}x{img.shape[0]}')
        except Exception as e:
            print(f'  {cid}: {e}')
except Exception as e:
    print(f'Live API failed: {e}')

# ── Fallback: use dataset val images ──
if len(frames) == 0:
    print('\nFalling back to dataset val images...')
    val_img_dir = Path(YOLO_DIR) / 'images' / 'val'
    from src.analytics.camera_network import CameraNetwork
    net = CameraNetwork()

    loaded = 0
    for img_path in sorted(val_img_dir.glob('*.jpg')):
        if loaded >= 9:
            break
        # Filename format: {camera_id}_{timestamp}.jpg  OR  {camera_id}_{date}_{time}.jpg
        cid = img_path.stem.split('_')[0]
        if cid in frames:
            continue
        img = cv2.imread(str(img_path))
        if img is not None:
            node = net.nodes.get(cid)
            frames[cid] = {
                'image': img,
                'lat': node.lat if node else 1.35,
                'lon': node.lon if node else 103.82,
            }
            print(f'  {cid} (dataset): {img.shape[1]}x{img.shape[0]}')
            loaded += 1

    if len(frames) == 0:
        # Last resort: load any 9 images and label them generically
        print('Scanning all val images...')
        seen_cids = set()
        for img_path in sorted(val_img_dir.glob('*.jpg')):
            if len(frames) >= 9:
                break
            img = cv2.imread(str(img_path))
            if img is not None:
                cid = img_path.stem.split('_')[0]
                if cid not in seen_cids:
                    frames[cid] = {'image': img, 'lat': 1.35, 'lon': 103.82}
                    seen_cids.add(cid)
                    print(f'  {cid}: {img.shape[1]}x{img.shape[0]}')

if len(frames) == 0:
    raise RuntimeError('No frames — check YOLO_DIR path in Cell 1')
print(f'\n{len(frames)} frames ready')

In [ ]:
# ── Cell 5: Fetch Live Weather ────────────────────────────────────────────
import urllib.request, json

def get_live_weather():
    try:
        data = json.loads(urllib.request.urlopen(
            'https://api.data.gov.sg/v1/environment/24-hour-weather-forecast', timeout=5
        ).read())
        return data['items'][0]['general']['forecast']
    except Exception:
        return 'unknown'

def get_live_temperature():
    try:
        data = json.loads(urllib.request.urlopen(
            'https://api.data.gov.sg/v1/environment/air-temperature', timeout=5
        ).read())
        readings = data['items'][0]['readings']
        return round(sum(r['value'] for r in readings) / len(readings), 1)
    except Exception:
        return 28.0

weather = get_live_weather()
temperature = get_live_temperature()
print(f'Weather: {weather}')
print(f'Temperature: {temperature}°C')

In [ ]:
# ── Cell 6: Run Inference on All Selected Cameras ─────────────────────────
import json
from datetime import datetime, timezone, timedelta

SGT = timezone(timedelta(hours=8))
timestamp = datetime.now(SGT).isoformat()

results = {}
for camera_id, frame_data in frames.items():
    try:
        result = pipeline.process_frame(
            image_bgr=frame_data['image'],
            camera_id=camera_id,
            timestamp=timestamp,
            weather=weather,
            temperature=temperature,
            frame_id=0,
        )
        results[camera_id] = result
        ts = result.traffic_state
        print(
            f'{camera_id:<6} {result.road:<5} {result.area:<20} '
            f'vehicles={ts.total_vehicles:<4} '
            f'occ={ts.occupancy*100:4.1f}% '
            f'LOS={ts.los.value} '
            f'congestion={ts.congestion_score:.2f} '
            f'[{ts.weather[:18]}]'
        )
    except Exception as e:
        print(f'{camera_id}: FAILED — {e}')

if not results:
    raise RuntimeError('All cameras failed — check pipeline setup')
print(f'\nInference complete. {len(results)} cameras. '
      f'Avg pipeline: {sum(r.pipeline_ms for r in results.values())/len(results):.0f}ms/frame')

In [ ]:
# ── Cell 7: Multi-Camera Speed Estimation ─────────────────────────────────
# Second pass — confirms tracks (min_hits=2), extracts embeddings, populates
# the re-ID gallery and fires cross-camera speed estimation.
#
# Key design: cameras are processed in road-order (upstream → downstream)
# with staggered timestamps (~45s apart) so travel_time > 0 when the
# downstream camera queries the upstream gallery.
#
# With static single frames the same physical vehicle won't appear in two
# cameras, so organic re-ID matches are rare. The synthetic demo below
# shows the full speed pipeline working end-to-end on a realistic
# adjacent-camera pair from the network.
from datetime import timedelta

TRAVEL_OFFSET_S = 45   # seconds — ~70 km/h across ~0.9 km adjacent cameras

# Sort selected cameras by road order: group by road, sort by lat/lon axis
from src.analytics.camera_network import CameraNetwork
_net = CameraNetwork()

def _road_order_key(cid):
    node = _net.nodes.get(cid)
    if node is None:
        return (cid, 0.0)
    road = node.road
    axis = _net._road_nodes.get(road, [])
    # Return (road, position along sort axis)
    sort_lat = road in ("CTE", "BKE", "KJE", "KPE", "NSC")
    return (road, node.lat if sort_lat else node.lon)

camera_ids_ordered = sorted(frames.keys(), key=_road_order_key)

print("Second pass — road-ordered, staggered timestamps:")
organic_readings = []
for i, camera_id in enumerate(camera_ids_ordered):
    frame_data = frames[camera_id]
    ts = (datetime.fromisoformat(timestamp) + timedelta(seconds=i * TRAVEL_OFFSET_S)).isoformat()
    result = pipeline.process_frame(
        image_bgr=frame_data['image'],
        camera_id=camera_id,
        timestamp=ts,
        weather=weather,
        temperature=temperature,
        frame_id=1,
    )
    results[camera_id] = result
    ts_label = result.traffic_state
    print(
        f"  {camera_id:<6} tracks={result.num_active_tracks:<3} "
        f"occ={ts_label.occupancy*100:4.1f}%  ts_offset=+{i*TRAVEL_OFFSET_S}s"
    )
    if result.speed_readings:
        for sr in result.speed_readings:
            organic_readings.append(sr)
            print(f"    ↳ SPEED (organic): {sr.camera_from}→{sr.camera_to}  "
                  f"{sr.speed_kmh:.1f} km/h  [{sr.congestion_band}]  "
                  f"(sim={sr.similarity:.3f})")

print()

# ── Synthetic speed demo: show system end-to-end on an adjacent camera pair ──
# Picks the first adjacent pair on the same road from our selected cameras.
# Uses realistic timestamps (T=camera_from pass, T+travel_time=camera_to pass).
print("Speed estimator demo (synthetic re-ID match on adjacent camera pair):")
from src.tracking.vehicle_reid import ReIDMatch
from src.analytics.speed_estimator import SpeedEstimator

# Find adjacent pairs among selected cameras
demo_readings = []
selected_set = set(frames.keys())
for edge in _net.edges:
    if edge.cam_a.camera_id in selected_set and edge.cam_b.camera_id in selected_set:
        # Realistic travel time: distance / 70 km/h
        travel_s = (edge.distance_km / 70.0) * 3600
        t_base = datetime.fromisoformat(timestamp)
        match = ReIDMatch(
            gallery_camera=edge.cam_a.camera_id,
            query_camera=edge.cam_b.camera_id,
            gallery_track_id=1,
            query_track_id=1,
            similarity=0.923,
            gallery_timestamp=t_base.isoformat(),
            query_timestamp=(t_base + timedelta(seconds=travel_s)).isoformat(),
            cls="car",
        )
        reading = pipeline.speed_estimator.estimate(match)
        if reading:
            demo_readings.append(reading)
            print(f"  {reading.camera_from}→{reading.camera_to}  "
                  f"{reading.road}  dist={reading.distance_km:.3f} km  "
                  f"travel={reading.travel_time_s:.0f}s  "
                  f"speed={reading.speed_kmh:.1f} km/h  "
                  f"[{reading.congestion_band}]  limit={reading.speed_limit} km/h")

if not demo_readings:
    print("  (no adjacent pairs among selected cameras)")

print()
print("Road speed profile:")
profile = pipeline.speed_estimator.road_speed_profile()
if profile:
    for road, info in profile.items():
        print(f"  {road:<6} {info['avg_speed_kmh']:>6.1f} km/h  "
              f"[{info['congestion_band']}]  "
              f"limit {info['speed_limit']} km/h  "
              f"({info['num_readings']} readings)")
else:
    print("  (no readings — needs real video for organic cross-camera matching)")


In [ ]:
# ── Cell 8: Annotated Output Frames ───────────────────────────────────────
import cv2
from IPython.display import display, Image as IPImage
import os

for camera_id, result in results.items():
    frame = frames[camera_id]['image']
    annotated = pipeline.draw(frame, result)

    out_path = f'{OUTPUT_DIR}/{camera_id}_{result.road}_{result.traffic_state.los.value}.jpg'
    cv2.imwrite(out_path, annotated)

    # Display inline (resize for Colab)
    h, w = annotated.shape[:2]
    scale = min(1.0, 900 / w)
    disp = cv2.resize(annotated, (int(w*scale), int(h*scale)))
    _, buf = cv2.imencode('.jpg', disp, [cv2.IMWRITE_JPEG_QUALITY, 85])
    print(f'Camera {camera_id} | {result.road} | {result.area} | LOS {result.traffic_state.los.value}')
    display(IPImage(data=buf.tobytes()))
    print()

In [ ]:
# ── Cell 9: Network Summary Dashboard ─────────────────────────────────────
import json
from src.analytics.traffic_analytics import TrafficAnalytics

all_states = [r.traffic_state for r in results.values()]
summary = TrafficAnalytics().network_summary(all_states)

print('=' * 60)
print('  SINGAPORE EXPRESSWAY NETWORK SUMMARY')
print('=' * 60)
print(f'  Cameras active:  {summary["total_cameras"]}')
print(f'  Total vehicles:  {summary["total_vehicles"]}')
print(f'  Avg occupancy:   {summary["avg_occupancy"]*100:.1f}%')
print(f'  Avg congestion:  {summary["avg_congestion"]:.3f}')
print(f'  Worst camera:    {summary["worst_camera"]}')
print()
print('  By Road:')
for road, info in summary['by_road'].items():
    print(f'    {road:<8} {info["total_vehicles"]:>4} vehicles  '
          f'congestion={info["avg_congestion"]:.2f}  '
          f'LOS {info["avg_los"]}')
print()
print('  By Region:')
for region, info in summary['by_region'].items():
    print(f'    {region:<10} {info["cameras"]} cameras  '
          f'{info["total_vehicles"]:>4} vehicles  '
          f'congestion={info["avg_congestion"]:.2f}')

# Save full results to Drive
with open(f'{OUTPUT_DIR}/network_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'\nSaved to {OUTPUT_DIR}/network_summary.json')

In [ ]:
# ── Cell 10: Per-Camera Detail ────────────────────────────────────────────
print(f'{"Cam":<6} {"Road":<6} {"Region":<10} {"Area":<22} {"Veh":>4} {"Occ%":>6} {"LOS":<4} {"Cong":>6} {"Tracks":>7}')
print('-' * 80)
for cid, result in sorted(results.items()):
    ts = result.traffic_state
    print(
        f'{cid:<6} {result.road:<6} {result.region:<10} {result.area:<22} '
        f'{ts.total_vehicles:>4} {ts.occupancy*100:>5.1f}% '
        f'{ts.los.value:<4} {ts.congestion_score:>6.3f} '
        f'{result.num_active_tracks:>7}'
    )